<a href="https://colab.research.google.com/github/syedahijabzahra/DevelopersHub-AI-ML-Internship/blob/main/Phase_2/Context_Aware_Chatbot_RAG/context_aware_chatbot_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Context-Aware Chatbot Using RAG (Retrieval-Augmented Generation)
# Document-based Q&A with Conversation Memory

!pip install langchain langchain-community langchain-text-splitters langchain-huggingface sentence-transformers faiss-cpu transformers -q

import warnings
warnings.filterwarnings('ignore')

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from transformers import pipeline

# Sample knowledge base (custom corpus about a fictional company's policies)
knowledge_base = """
Our company, TechCorp Solutions, was founded in 2015 and specializes in cloud computing services.
We offer three subscription plans: Basic ($10/month), Pro ($25/month), and Enterprise ($99/month).

The Basic plan includes 10GB storage and email support.
The Pro plan includes 100GB storage, priority email support, and API access.
The Enterprise plan includes unlimited storage, 24/7 phone support, dedicated account manager, and custom integrations.

Our refund policy allows customers to request a full refund within 14 days of purchase.
After 14 days, refunds are evaluated on a case-by-case basis.

For technical support, customers can reach us via email at support@techcorp.com or through live chat on our website.
Our support team operates Monday to Friday, 9 AM to 6 PM EST for Basic and Pro plans.
Enterprise customers receive 24/7 support including weekends and holidays.

To upgrade your plan, log into your account dashboard and navigate to the Billing section.
Changes take effect immediately and you will be charged a prorated amount for the current billing cycle.

We use industry-standard encryption (AES-256) to protect all customer data.
All data is backed up daily and stored across multiple geographic regions for redundancy.
"""

# Split text into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=30)
chunks = text_splitter.split_text(knowledge_base)
print(f"Created {len(chunks)} text chunks")

# Create embeddings and vector store
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_store = FAISS.from_texts(chunks, embeddings)
print("✅ Vector store created")

# Load a lightweight generation model
from transformers import T5Tokenizer, T5ForConditionalGeneration

tokenizer_gen = T5Tokenizer.from_pretrained("google/flan-t5-base")
model_gen = T5ForConditionalGeneration.from_pretrained("google/flan-t5-base")

def generate_response(prompt, max_length=150):
    inputs = tokenizer_gen(prompt, return_tensors="pt", truncation=True, max_length=512)
    outputs = model_gen.generate(**inputs, max_length=max_length)
    return tokenizer_gen.decode(outputs[0], skip_special_tokens=True)

# Conversation memory
conversation_history = []

def retrieve_context(query, k=2):
    docs = vector_store.similarity_search(query, k=k)
    return "\n".join([doc.page_content for doc in docs])

def chat_with_rag(query):
    # Retrieve relevant context
    context = retrieve_context(query)

    # Build prompt with context and conversation history
    history_text = ""
    if conversation_history:
        history_text = "Previous conversation:\n" + "\n".join(conversation_history[-2:]) + "\n\n"

    prompt = f"{history_text}Context: {context}\n\nQuestion: {query}\nAnswer:"

    # Generate response
    response = generate_response(prompt, max_length=150)s

    # Update conversation history
    conversation_history.append(f"Q: {query}")
    conversation_history.append(f"A: {response}")

    return response, context

# Test conversations
test_questions = [
    "What subscription plans do you offer?",
    "How much does the Pro plan cost?",
    "What is your refund policy?",
    "How do I upgrade my plan?",
    "What support do Enterprise customers get?"
]

print("\n" + "="*60)
print("CONTEXT-AWARE CHATBOT - DEMO CONVERSATION")
print("="*60 + "\n")

for question in test_questions:
    answer, retrieved_context = chat_with_rag(question)
    print(f"🙋 User: {question}")
    print(f"📚 Retrieved Context: {retrieved_context[:100]}...")
    print(f"🤖 Bot: {answer}")
    print("-"*60)

print("\n✅ RAG-based chatbot demo complete!")
print(f"Total conversation turns: {len(test_questions)}")

Created 8 text chunks


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ Vector store created


tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]


CONTEXT-AWARE CHATBOT - DEMO CONVERSATION

🙋 User: What subscription plans do you offer?
📚 Retrieved Context: Our company, TechCorp Solutions, was founded in 2015 and specializes in cloud computing services.
We...
🤖 Bot: Basic ($10/month), Pro ($25/month), and Enterprise ($99/month)
------------------------------------------------------------
🙋 User: How much does the Pro plan cost?
📚 Retrieved Context: The Basic plan includes 10GB storage and email support.
The Pro plan includes 100GB storage, priorit...
🤖 Bot: 25/month
------------------------------------------------------------
🙋 User: What is your refund policy?
📚 Retrieved Context: Our refund policy allows customers to request a full refund within 14 days of purchase.
After 14 day...
🤖 Bot: allows customers to request a full refund within 14 days of purchase
------------------------------------------------------------
🙋 User: How do I upgrade my plan?
📚 Retrieved Context: To upgrade your plan, log into your account dashboard and 